# M6.5 · torch.compile, FX ve operatör füzyonu

Derleyicinin gözünden PyTorch. Füzyonun bellek trafiğini nasıl azalttığını elle hesaplıyor, saf Python'da torch.fx benzeri bir izleyici (tracer) ile bir fonksiyonu grafa çevirip CSE ve ölü kod eleme geçişleri yazıyor, NumPy'da elementwise işlemleri füzyon gruplarına ayırıp taşınan baytı modelliyor, PyTorch'ta FX ile Conv + BatchNorm katlama yapıyor ve en sonda `torch.compile` ile graph break'leri bulup gideriyor, füzyonun kazancını ölçüyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m6-5-compile.html](https://m-gonul.github.io/pytorch-lab/m6-5-compile.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m6_5_compile.py
CHECKS = {
 "b": "import math\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n    tanh, exp = need(ns, \"tanh\"), need(ns, \"exp\")\n\n    def f(x, y):\n        a = tanh(x * y)\n        b = tanh(y * x)\n        unused = exp(x)\n        return a + b * 2 - 1\n\n    def tr():\n        g = need(ns, \"trace\")(f, 2)\n        ops = [op for _, op, _ in g.nodes]\n        ensure(ops.count(\"input\") == 2 and ops.count(\"tanh\") == 2 and ops.count(\"exp\") == 1, f\"düğümler {ops}\")\n        close(need(ns, \"run\")(g, [0.3, 0.5]), f(0.3, 0.5))\n        close(ns[\"run\"](g, [-1.2, 0.7]), f(-1.2, 0.7))\n    R.test(\"trace + run = Python fonksiyonu\", tr)\n\n    def cs():\n        g = need(ns, \"cse\")(ns[\"trace\"](f, 2))\n        ops = [op for _, op, _ in g.nodes]\n        ensure(ops.count(\"tanh\") == 1 and ops.count(\"mul\") == 2, f\"CSE sonrası {ops}: x*y ve y*x tek düğüm olmalı\")\n        close(ns[\"run\"](g, [0.3, 0.5]), f(0.3, 0.5))\n    R.test(\"cse (değişmeli işlemler dahil)\", cs)\n\n    def dc():\n        g = need(ns, \"dce\")(ns[\"cse\"](ns[\"trace\"](f, 2)))\n        ops = [op for _, op, _ in g.nodes]\n        ensure(\"exp\" not in ops and ops.count(\"input\") == 2, f\"DCE sonrası {ops}\")\n        close(ns[\"run\"](g, [0.3, 0.5]), f(0.3, 0.5))\n        before = len(ns[\"trace\"](f, 2).nodes)\n        return f\"{before} düğüm → {len(g.nodes)} düğüm\"\n    R.test(\"dce\", dc)\n\n    def sub():\n        g = need(ns, \"cse\")(ns[\"trace\"](lambda x, y: (x - y) * (y - x), 2))\n        close(ns[\"run\"](g, [3.0, 1.0]), -4.0, what=\"sub değişmeli değil\")\n    R.test(\"sub değişmeli sayılmıyor\", sub)\n",
 "c": "import numpy as np\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    N = 10 ** 6\n    G = [(\"x\", \"input\", ()), (\"a\", \"input\", ()), (\"b\", \"input\", ()),\n         (\"t1\", \"mul\", (\"x\", \"a\")), (\"t2\", \"add\", (\"t1\", \"b\")), (\"y\", \"tanh\", (\"t2\",))]\n    nb = {\"x\": 4 * N, \"a\": 0, \"b\": 0, \"t1\": 4 * N, \"t2\": 4 * N, \"y\": 4 * N}\n\n    def ev():\n        rng = np.random.default_rng(0)\n        inp = {\"x\": rng.normal(size=5), \"a\": 2.0, \"b\": -1.0}\n        close(need(ns, \"evaluate\")(G, inp)[\"y\"], np.tanh(inp[\"x\"] * 2 - 1))\n    R.test(\"evaluate\", ev)\n\n    def fs():\n        groups = need(ns, \"fuse\")(G, [\"y\"])\n        ensure([sorted(g) for g in groups] == [[\"t1\", \"t2\", \"y\"]], f\"gruplar {groups}\")\n        ensure(need(ns, \"bytes_moved\")(G, [[\"t1\"], [\"t2\"], [\"y\"]], nb, [\"y\"]) == 24 * N, \"füzyonsuz 24 MB\")\n        ensure(ns[\"bytes_moved\"](G, groups, nb, [\"y\"]) == 8 * N, \"füzyonlu 8 MB\")\n    R.test(\"elle örnek: 24 MB → 8 MB\", fs)\n\n    G2 = [(\"x\", \"input\", ()), (\"e\", \"exp\", (\"x\",)), (\"s\", \"sum\", (\"e\",)), (\"m\", \"mul\", (\"e\", \"x\")),\n          (\"t\", \"tanh\", (\"m\",)), (\"u\", \"add\", (\"t\", \"e\")), (\"o\", \"exp\", (\"u\",))]\n    nb2 = {k: 4 * N for k, _, _ in G2}; nb2[\"s\"] = 4\n\n    def fs2():\n        groups = [sorted(g) for g in need(ns, \"fuse\")(G2, [\"o\", \"s\"])]\n        ensure([\"e\"] in groups and [\"s\"] in groups, f\"çok tüketicili e ve indirgeme s ayrı grup: {groups}\")\n        ensure([\"m\", \"o\", \"t\", \"u\"] in groups, f\"m → t → u → o tek grup: {groups}\")\n        unf = need(ns, \"bytes_moved\")(G2, [[n] for n, op, _ in G2 if op != \"input\"], nb2, [\"o\", \"s\"])\n        fus = ns[\"bytes_moved\"](G2, ns[\"fuse\"](G2, [\"o\", \"s\"]), nb2, [\"o\", \"s\"])\n        R.table(\"Çok tüketicili graf\", [\"\", \"bayt\"], [[\"füzyonsuz\", f\"{unf:,}\"], [\"füzyonlu\", f\"{fus:,}\"]])\n        ensure(fus < unf, \"füzyon trafiği azaltmalı\")\n    R.test(\"çok tüketicili ve indirgemeli graf\", fs2)\n",
 "d": "import torch\nfrom torch import nn\nfrom labcheck import close, need, ensure\n\n\ndef randomize_bn(m):\n    g = torch.Generator().manual_seed(1)\n    for mod in m.modules():\n        if isinstance(mod, nn.BatchNorm2d):\n            mod.running_mean.copy_(torch.randn(mod.num_features, generator=g))\n            mod.running_var.copy_(torch.rand(mod.num_features, generator=g) + 0.5)\n            with torch.no_grad():\n                mod.weight.copy_(torch.randn(mod.num_features, generator=g))\n                mod.bias.copy_(torch.randn(mod.num_features, generator=g))\n\n\ndef check(ns, R):\n    def fold():\n        conv = nn.Conv2d(1, 1, 1); bn = nn.BatchNorm2d(1, eps=0.0).eval()\n        with torch.no_grad():\n            conv.weight.fill_(2.0); conv.bias.fill_(1.0)\n            bn.running_mean.fill_(3.0); bn.running_var.fill_(4.0); bn.weight.fill_(0.5); bn.bias.fill_(1.0)\n        f = need(ns, \"fold_conv_bn\")(conv, bn)\n        close([f.weight.item(), f.bias.item()], [0.5, 0.5], what=\"elle örnek\")\n    R.test(\"fold_conv_bn (elle örnek)\", fold)\n\n    def model():\n        torch.manual_seed(0)\n        m = nn.Sequential(nn.Conv2d(3, 8, 3, padding=1), nn.BatchNorm2d(8), nn.ReLU(),\n                          nn.Conv2d(8, 8, 3, stride=2, bias=False), nn.BatchNorm2d(8), nn.ReLU(), nn.BatchNorm2d(8)).double()\n        randomize_bn(m); m.eval()\n        x = torch.randn(4, 3, 20, 20, dtype=torch.float64)\n        fm = need(ns, \"fuse_model\")(m)\n        close(fm(x), m(x))\n        n_bn = sum(isinstance(mod, nn.BatchNorm2d) for mod in fm.modules())\n        ensure(n_bn == 1, f\"iki BN katlanmalı, ReLU'dan sonraki BN kalmalı; kalan BN: {n_bn}\")\n        calls = [n.target for n in fm.graph.nodes if n.op == \"call_module\"]\n        return f\"FX grafı: {calls}\"\n    R.test(\"fuse_model: eval çıktısı aynı, 2 BN katlandı\", model)\n\n    def shared():\n        class Two(nn.Module):\n            def __init__(self):\n                super().__init__(); self.c = nn.Conv2d(2, 2, 1); self.bn = nn.BatchNorm2d(2)\n\n            def forward(self, x):\n                y = self.c(x)\n                return self.bn(y) + y\n        torch.manual_seed(0)\n        m = Two().double(); randomize_bn(m); m.eval()\n        x = torch.randn(1, 2, 4, 4, dtype=torch.float64)\n        close(need(ns, \"fuse_model\")(m)(x), m(x), what=\"conv çıktısı başka yerde de kullanılıyorsa katlama yapılmamalı\")\n    R.test(\"çok kullanıcılı conv'a dokunulmuyor\", shared)\n",
 "e": "import logging\nimport torch\nfrom labcheck import close, need, ensure, bench, fmt_time\n\ntorch._logging.set_logs(dynamo=logging.ERROR)\n\n\ndef check(ns, R):\n    def br():\n        x = torch.randn(1000)\n        a = need(ns, \"count_breaks\")(ns[\"branchy\"], x)\n        b = ns[\"count_breaks\"](need(ns, \"branchy_fixed\"), x)\n        ensure(a >= 1, f\"branchy'de en az bir graph break bekleniyordu ({a})\")\n        ensure(b == 0, f\"branchy_fixed'de {b} graph break var\")\n        return f\"branchy: {a} break, branchy_fixed: {b}\"\n    R.test(\"graph break'ler sayılıyor ve gideriliyor\", br)\n\n    def same():\n        for s in (1.0, -1.0):\n            x = torch.rand(50) * s\n            close(need(ns, \"branchy_fixed\")(x), ns[\"branchy\"](x), what=f\"işaret {s:+.0f}\")\n    R.test(\"branchy_fixed = branchy (iki dal)\", same)\n\n    def comp():\n        torch._dynamo.reset()\n        x = torch.randn(3000)\n        y_c = need(ns, \"compiled_chain\")(x)\n        close(y_c, need(ns, \"chain\")(x), atol=1e-5)\n    R.test(\"compiled_chain = chain\", comp)\n    if \"compiled_chain\" not in ns:\n        return\n    x = torch.randn(4_000_000)\n    ns[\"compiled_chain\"](x)\n    t_e = bench(lambda: ns[\"chain\"](x), repeat=5)\n    t_c = bench(lambda: ns[\"compiled_chain\"](x), repeat=5)\n    R.table(\"4M eleman, elementwise zincir (CPU)\", [\"\", \"süre\"], [[\"eager\", fmt_time(t_e)], [\"torch.compile (Inductor)\", fmt_time(t_c)]],\n            note=\"Eager her işlem için ayrı bir döngü ve ara tensör; Inductor tek bir C++ döngüsü üretir. Kazanç, işlem bellek-bağımlıyken büyük.\")\n"
}

In [ ]:
import importlib, labcheck, checks_m6_5_compile as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

`torch.compile(fn)` üç katmandan oluşur:

1. **Dynamo** — Python bytecode'unu çalışma anında yakalar, tensör işlemlerini bir FX grafına koyar. Yakalayamadığı (ör. veriye bağlı Python akışı, `print`, `.item()`) yerde graph break olur: graf ikiye bölünür, araya normal Python girer.
2. **AOTAutograd** — ileri ve geri grafını önceden çıkarır.
3. **Inductor** — grafı optimize eder, elementwise işlemleri operatör füzyonuyla birleştirir, GPU'da Triton, CPU'da C++ kodu üretir.

Elementwise işlemler **bellek-bağımlıdır**: bir `add`, okuduğu her 8 bayt için 1 FLOP yapar. Üç işlemi zincirlemek her ara sonucu belleğe yazıp geri okumak demek; füzyonla girdi bir kez okunur, çıktı bir kez yazılır.

**torch.fx** grafı Python'dan erişilebilir kılar: `symbolic_trace(model)` her işlemi bir `Node` (`placeholder`, `call_module`, `call_function`, `output`) olarak kaydeder; düğümleri değiştirip `GraphModule` olarak yeniden derleyebilirsin.

## 1 · Elle

Füzyonda ara değerler $x\cdot a$ ve $x\cdot a+b$ hiç belleğe inmez. Bellek-bağımlı bir işlemde süre $\approx$ bayt / bant genişliği olduğundan kazanç bayt oranı kadar: 3 kat.

Katlanmış konvolüsyon: $\ w'x+b'=0.5x+0.5$; orijinal $\ \gamma\frac{(2x+1)-3}{2}+1=0.5x+0.5$. Aynı.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**$y=\operatorname{gelu}(x\cdot a+b)$, $x$: $10^6$ fp32 eleman, $a,b$ skaler (bellek trafiğine katkısı yok)**

- `unf`: Füzyonsuz (mul, add, gelu ayrı kernel): okunan + yazılan bayt
- `fus`: Tek füzyonlu kernel: bayt
- `ratio`: Bellek-bağımlı işlemde beklenen hız kazancı üst sınırı

**Conv + BN katlama: tek kanal, $w=2$, $b=1$; BN: $\mu=3$, $\sigma^2=4$, $\gamma=0.5$, $\beta=1$, $\epsilon=0$**

- `wf`: Katlanmış ağırlık $w'=w\gamma/\sqrt{\sigma^2+\epsilon}$
- `bf`: Katlanmış bias $b'=(b-\mu)\gamma/\sqrt{\sigma^2+\epsilon}+\beta$

In [ ]:
cevaplar = dict(
    unf=None,
    fus=None,
    ratio=None,
    wf=None,
    bf=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"unf":[24000000,0],"fus":[8000000,0],"ratio":[3,0],"wf":[0.5,0],"bf":[0.5,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, torch.fx'in minik bir kopyası. `Graph` hazır: `nodes` = `(ad, işlem, argümanlar)` listesi, `add(op, args)` yeni düğüm adı döndürür, `output` çıkış düğümünün adı. İşlemler: `"input"`, `"add"`, `"sub"`, `"mul"`, `"tanh"`, `"exp"`. Argümanlar düğüm adları (`str`) ya da sabit sayılar.

- `Proxy(graph, name)` — `+`, `-`, `*` (sağdan sayılarla da) yeni düğüm kaydeder ve yeni `Proxy` döndürür.
- `tanh(v)`, `exp(v)` — `Proxy` ise düğüm kaydet, sayıysa `math` ile hesapla.
- `trace(fn, n_inputs)` → `Graph`.
- `run(graph, inputs)` → çıkış değeri.
- `cse(graph)` — aynı `(işlem, argümanlar)` düğümlerini tek düğümde birleştir (`add` ve `mul` değişmeli: argüman sırası önemsiz), kullanımları yeniden yönlendir.
- `dce(graph)` — çıkışa ulaşmayan düğümleri sil (girdiler kalır).

In [ ]:
%%writefile stage_b.py
import math


class Graph:
    def __init__(self):
        self.nodes = []
        self.output = None

    def add(self, op, args):
        name = f"v{len(self.nodes)}"
        self.nodes.append((name, op, tuple(args)))
        return name


# ---- senin kodun ----

class Proxy:
    def __init__(self, graph, name):
        self.graph, self.name = graph, name

    def __add__(self, o):
        ...

    __radd__ = __add__

    def __sub__(self, o):
        ...

    def __mul__(self, o):
        ...

    __rmul__ = __mul__


def tanh(v):
    ...


def exp(v):
    ...


def trace(fn, n_inputs):
    ...


def run(graph, inputs):
    ...


def cse(graph):
    ...


def dce(graph):
    ...


def f(x, y):
    a = tanh(x * y)
    b = tanh(y * x)          # aynı ifade (değişmeli)
    unused = exp(x)          # kullanılmıyor
    return a + b * 2
g = trace(f, 2)
print(len(g.nodes), run(g, [0.3, 0.5]), f(0.3, 0.5))
g = dce(cse(g))
print(len(g.nodes), run(g, [0.3, 0.5]))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy; graf formatı b'deki gibi `(ad, işlem, argümanlar)` listesi, işlemler: `"input"`, `"add"`, `"mul"`, `"tanh"`, `"exp"` (elementwise) ve `"sum"` (indirgeme, füzyon sınırı).

- `evaluate(graph, inputs)` → her düğümün değeri (sözlük).
- `fuse(graph, outputs)` → gruplar listesi (düğüm adı listeleri, `input`'lar hariç). Elementwise bir düğüm, elementwise girdisinin grubuna katılır **eğer** o girdinin tek tüketicisi bu düğümse ve girdi bir çıkış değilse. `sum` kendi grubunda.
- `bytes_moved(graph, groups, nbytes, outputs)` — her grup, dışarıdan okuduğu değerlerin baytını okur ve grubun dışında kullanılan (ya da çıkış olan) değerlerinin baytını yazar. `nbytes[ad]` değer başına bayt.

In [ ]:
%%writefile stage_c.py
import numpy as np


def evaluate(graph, inputs):
    ...


def fuse(graph, outputs):
    ...


def bytes_moved(graph, groups, nbytes, outputs):
    ...


G = [("x", "input", ()), ("a", "input", ()), ("b", "input", ()),
     ("t1", "mul", ("x", "a")), ("t2", "add", ("t1", "b")), ("y", "tanh", ("t2",))]
N = 10 ** 6
nb = {"x": 4 * N, "a": 0, "b": 0, "t1": 4 * N, "t2": 4 * N, "y": 4 * N}
groups = fuse(G, ["y"])
print(groups)                                                   # [['t1', 't2', 'y']]
print(bytes_moved(G, [["t1"], ["t2"], ["y"]], nb, ["y"]), bytes_moved(G, groups, nb, ["y"]))   # 24e6, 8e6

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

- `fold_conv_bn(conv, bn)` → yeni `nn.Conv2d` (bias'lı): $\ s=\gamma/\sqrt{\sigma^2+\epsilon}$, $\ W'=W\cdot s$ (çıkış kanalı başına), $\ b'=(b-\mu)s+\beta$ (`conv.bias` yoksa $b=0$).
- `fuse_model(model)` → `torch.fx.GraphModule`: `symbolic_trace`; her `call_module` BN düğümünün girdisi bir Conv2d `call_module` düğümüyse ve conv çıktısının başka kullanıcısı yoksa, conv modülünü katlanmış olanla değiştir, BN düğümünü conv düğümüyle değiştir (`replace_all_uses_with`), BN düğümünü sil; `graph.lint()`, `recompile()`.

Kontrol modeli **eval modunda** karşılaştırır.

In [ ]:
%%writefile stage_d.py
import torch
import torch.fx as fx
from torch import nn


def fold_conv_bn(conv, bn):
    ...


def fuse_model(model):
    ...


torch.manual_seed(0)
m = nn.Sequential(nn.Conv2d(3, 8, 3, padding=1), nn.BatchNorm2d(8), nn.ReLU(), nn.Conv2d(8, 8, 3, bias=False), nn.BatchNorm2d(8)).eval()
fm = fuse_model(m)
x = torch.randn(2, 3, 16, 16)
print((fm(x) - m(x)).abs().max(), [type(mod).__name__ for mod in fm.modules()])

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

- `count_breaks(fn, *args)` — `torch._dynamo.explain(fn)(*args).graph_break_count`.
- `branchy_fixed(x)` — aşağıdaki fonksiyonla aynı sonucu veren, **graph break'siz** sürüm (`torch.where`):

```python
def branchy(x):
    out = x * 2
    if out.sum().item() > 0:
        out = out.tanh()
    else:
        out = out.sigmoid()
    return out + 1
```

- `chain(x)` — 8 ucuz elementwise işlemden (`*`, `+`, `-`) oluşan bir zincir; `compiled_chain = torch.compile(chain)`. Ucuz işlemler bellek-bağımlıdır, füzyonun etkisi en net bunlarda görünür. (`sin`, `exp` ağırlıklı zincirler CPU'da hesap-bağımlı olabilir; orada kazanç küçük, hatta negatif olabilir.)

İlk `torch.compile` çağrısı derleme yapar (CPU'da onlarca saniye sürebilir); kontrol sonra eager ile karşılaştırır.

In [ ]:
%%writefile stage_e.py
import torch


def count_breaks(fn, *args):
    ...


def branchy(x):
    out = x * 2
    if out.sum().item() > 0:
        out = out.tanh()
    else:
        out = out.sigmoid()
    return out + 1


def branchy_fixed(x):
    ...


def chain(x):
    ...


compiled_chain = torch.compile(chain)

x = torch.randn(1000)
print(count_breaks(branchy, x), count_breaks(branchy_fixed, x))   # ≥1, 0

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Kaynaklar

- [Ansel ve ark. — PyTorch 2: Faster Machine Learning Through Dynamic Python Bytecode Transformation and Graph Compilation](https://pytorch.org/assets/pytorch2-2.pdf) — ©
- [PyTorch Docs — torch.fx](https://docs.pytorch.org/docs/stable/fx.html) — BSD-3-Clause
- [PyTorch Tutorials — (beta) Building a Convolution/Batch Norm fuser in FX](https://docs.pytorch.org/tutorials/intermediate/fx_conv_bn_fuser.html) — BSD-3-Clause
- [PyTorch Tutorials — Introduction to torch.compile](https://docs.pytorch.org/tutorials/intermediate/torch_compile_tutorial.html) — BSD-3-Clause
- [Horace He — Making Deep Learning Go Brrrr From First Principles](https://horace.io/brrr_intro.html) — ©